# Gated Recurrent Unit (GRU) - TensorFlow / Keras

In [2]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# -----------------------------
# Reproducibility: Set seeds
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")

# -----------------------------
# Generate synthetic dataset
# -----------------------------
samples, seq_len, features = 1400, 30, 3
X = np.random.normal(size=(samples, seq_len, features)).astype("float32")

signal = (
    X[:, -10:, 0].mean(axis=1) +
    0.5 * X[:, :10, 1].mean(axis=1)
)
y = (signal > 0.05).astype("int64")

# Train/test split
X_train, X_test = X[:1000], X[1000:]
y_train, y_test = y[:1000], y[1000:]

# -----------------------------
# Define GRU model
# -----------------------------
model = keras.Sequential([
    layers.Input(shape=(seq_len, features)),
    layers.GRU(32),
    layers.Dropout(0.2),
    layers.Dense(2, activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.AdamW(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# -----------------------------
# Train model
# -----------------------------
model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=64
    )


TensorFlow version: 2.20.0


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 32)             │         3,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,618 (14.13 KB)

 Trainable params: 3,618 (14.13 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - accuracy: 0.5440 - loss: 0.6935 - val_accuracy: 0.6025 - val_loss: 0.6597
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6510 - loss: 0.6295 - val_accuracy: 0.7200 - val_loss: 0.5947
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7410 - loss: 0.5671 - val_accuracy: 0.8000 - val_loss: 0.5169
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7810 - loss: 0.4761 - val_accuracy: 0.8100 - val_loss: 0.4395
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8010 - loss: 0.4344 - val_accuracy: 0.8175 - val_loss: 0.4017
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8030 - loss: 0.4071 - val_accuracy: 0.8300 - val_loss: 0.3847
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8150 - loss: 0.3941 - val_accuracy: 0.8275 - val_loss: 0.3805
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8120 - loss: 0.3865 - val_accuracy: 0.8350 - v

In [3]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense

# -----------------------------
# 1. Load Dataset
# -----------------------------
vocab_size = 10000   # Top 10,000 words
max_length = 200     # Maximum review length

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

# -----------------------------
# 2. Padding Sequences
# -----------------------------
x_train = pad_sequences(x_train, maxlen=max_length, padding='post')
x_test = pad_sequences(x_test, maxlen=max_length, padding='post')

# -----------------------------
# 3. Build GRU Model
# -----------------------------
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    GRU(64, return_sequences=False),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
    ])

# -----------------------------
# 4. Compile Model
# -----------------------------
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# -----------------------------
# 5. Model Summary
# -----------------------------
model.summary()

# -----------------------------
# 6. Train Model
# -----------------------------
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

# -----------------------------
# 7. Evaluate Model
# -----------------------------
test_loss, test_accuracy = model.evaluate(x_test, y_test)

print(f"\nTest Accuracy: {test_accuracy:.4f}")

# -----------------------------
# 8. Prediction Example
# -----------------------------
sample_review = x_test[0]

prediction = model.predict(sample_review.reshape(1, -1))

if prediction[0][0] > 0.5:
    print("Positive Review")
else:
    print("Negative Review")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 96s 299ms/step - accuracy: 0.5627 - loss: 0.6617 - val_accuracy: 0.6168 - val_loss: 0.6051
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 142s 301ms/step - accuracy: 0.8547 - loss: 0.3332 - val_accuracy: 0.8800 - val_loss: 0.2888
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 139s 291ms/step - accuracy: 0.9295 - loss: 0.1910 - val_accuracy: 0.8854 - val_loss: 0.2972
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 142s 291ms/step - accuracy: 0.9522 - loss: 0.1350 - val_accuracy: 0.8758 - val_loss: 0.3497
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 91s 289ms/step - accuracy: 0.9679 - loss: 0.0932 - val_accuracy: 0.8636 - val_loss: 0.4589
782/782 ━━━━━━━━━━━━━━━━━━━━ 25s 32ms/step - accuracy: 0.8535 - loss: 0.4946

Test Accuracy: 0.8535
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step
Negative Review
